In [1]:
import os



In [2]:
pwd

'c:\\Users\\Revanth L\\OneDrive\\Pictures\\Desktop\\AI\\DL\\Project\\Kidney_Disease-Classification-Deep-Learning-Project\\research'

In [3]:
%cd "C:/Users/Revanth L/OneDrive/Pictures/Desktop/AI/DL/Project/Kidney_Disease-Classification-Deep-Learning-Project"

C:\Users\Revanth L\OneDrive\Pictures\Desktop\AI\DL\Project\Kidney_Disease-Classification-Deep-Learning-Project


c:\Users\Revanth L\anaconda3\envs\kidney\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
pwd

'C:\\Users\\Revanth L\\OneDrive\\Pictures\\Desktop\\AI\\DL\\Project\\Kidney_Disease-Classification-Deep-Learning-Project'

In [17]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return prepare_base_model_config


In [19]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [34]:
from pathlib import Path
import tensorflow as tf


class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def get_base_model(self):
        self.model = tf.keras.applications.ResNet50(
            input_shape=self.config.params_image_size,
            include_top=self.config.params_include_top,
            weights=self.config.params_weights
        )
        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        gap = tf.keras.layers.GlobalAveragePooling2D()(
            model.output
        )

        prediction = tf.keras.layers.Dense(units=classes, activation='softmax')(gap)
        full_model = tf.keras.models.Model(inputs=model.input, outputs=prediction)
        full_model.compile( optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
        full_model.summary()
        return full_model



    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

In [35]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()

    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-06-22 12:35:00,005: INFO: common yaml file: config\config.yaml loaded successfully]
[2026-06-22 12:35:00,016: INFO: common yaml file: params.yaml loaded successfully]


[2026-06-22 12:35:00,022: INFO: common created directory at: artifacts]
[2026-06-22 12:35:00,028: INFO: common created directory at: artifacts/prepare_base_model]
[2026-06-22 12:35:02,561: WARNING: saving_utils Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_13 (InputLayer)          [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_13[0][0]']               
                          